# Buffer adjustment comparison (multi-run)

Inspect how DBR buffer targets evolve across multiple experiments with different `buffers_update_multiplyer` values.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import yaml
from IPython.display import display

from manusim.metrics import ExperimentMetrics

In [ ]:
# Update labels if needed after reading each run config.
EXPERIMENTS = {
    "m?_run0": Path(r"C:/github/master/manufacturing-scheduling/data/experiments/simulation/2606012304_dbr_0"),
    "m?_run1": Path(r"C:/github/master/manufacturing-scheduling/data/experiments/simulation/2606012304_dbr_1"),
    "m?_run2": Path(r"C:/github/master/manufacturing-scheduling/data/experiments/simulation/2606012304_dbr_2"),
    "m?_run3": Path(r"C:/github/master/manufacturing-scheduling/data/experiments/simulation/2606012304_dbr_3"),
}
RUN = 1
DISCARD_LAST_TIME_FRACTION = 0.5  # keep first 50% of timesteps per run

for label, path in EXPERIMENTS.items():
    print(f"{label}: {path} -> {'OK' if path.exists() else 'MISSING'}")

In [ ]:
def discard_last_timesteps(logs: pd.DataFrame, fraction: float) -> pd.DataFrame:
    """Drop rows in the last `fraction` of the simulation time span."""
    if logs.empty or "time" not in logs.columns or fraction <= 0:
        return logs
    t_min = logs["time"].min()
    t_max = logs["time"].max()
    if t_max <= t_min:
        return logs
    cutoff = t_min + (1.0 - fraction) * (t_max - t_min)
    return logs.loc[logs["time"] <= cutoff].copy()


def load_experiment_logs(experiment_path: Path, run: int = 1):
    cfg_path = experiment_path / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        raise FileNotFoundError(f"Missing config: {cfg_path}")
    config = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))

    metrics = ExperimentMetrics(experiment_path)
    metrics.read_logs(metrics=["constraintBufferTarget", "constraintBufferLevel", "constraintBufferQueue", "constraintBufferWip", "shippingBufferTarget", "shippingBufferLevel"])
    logs = metrics.logs.copy()

    if "run" in logs.columns:
        logs = logs[logs["run"] == run]
    elif "run_id" in logs.columns:
        logs = logs[logs["run_id"] == run]

    logs = discard_last_timesteps(logs, DISCARD_LAST_TIME_FRACTION)

    sim_cfg = config.get("simulation", {})
    return logs, config, {
        "buffers_update_multiplyer": sim_cfg.get("buffers_update_multiplyer"),
        "buffers_update_warmup": sim_cfg.get("buffers_update_warmup"),
        "cb_target_level": sim_cfg.get("cb_target_level"),
        "cb_update": sim_cfg.get("cb_update"),
        "sb_update": sim_cfg.get("sb_update"),
    }


def target_changes(logs: pd.DataFrame, variable: str) -> pd.DataFrame:
    df = logs[logs["variable"] == variable].copy()
    if df.empty:
        return df
    df = df.sort_values(["key", "time"]).copy()
    df["prev"] = df.groupby("key")["value"].shift(1)
    return df[df["prev"].isna() | (df["value"] != df["prev"])]

In [ ]:
all_logs = []
summary_rows = []

for label, exp_path in EXPERIMENTS.items():
    logs, config, controls = load_experiment_logs(exp_path, RUN)
    logs = logs.copy()
    logs["experiment"] = label
    logs["multiplier"] = controls["buffers_update_multiplyer"]
    all_logs.append(logs)

    summary_rows.append({
        "experiment": label,
        "path": str(exp_path),
        **controls,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("buffers_update_multiplyer")
display(summary_df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

logs_df = pd.concat(all_logs, ignore_index=True)
logs_df["multiplier_label"] = logs_df["multiplier"].map(lambda m: f"multiplier={m}")

cb_df = logs_df[logs_df["variable"] == "constraintBufferTarget"].copy()
if not cb_df.empty:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=cb_df,
        x="time",
        y="value",
        hue="multiplier_label",
        palette="tab10"
    )
    plt.title("Constraint buffer target")
    plt.xlabel("Time")
    plt.ylabel("Buffer Target Value")
    plt.legend(title="Multiplier")
    plt.tight_layout()
    plt.show()

sb_df = logs_df[logs_df["variable"] == "shippingBufferTarget"].copy()
for product in sorted(sb_df["key"].dropna().unique()):
    product_df = sb_df[sb_df["key"] == product]
    if product_df.empty:
        continue
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=product_df,
        x="time",
        y="value",
        hue="multiplier_label",
        palette="tab10",
    )
    plt.title(f"Shipping buffer target — {product}")
    plt.xlabel("Time")
    plt.ylabel("Buffer Target Value")
    plt.legend(title="Multiplier")
    plt.tight_layout()
    plt.show()

In [ ]:
# Select only the last half (time >= 2.5M) for each group, then aggregate
from IPython.display import display_html

summary_list = []
for (variable, key, multiplier), group in logs_df.groupby(["variable", "key", "multiplier"]):
    time_cutoff = 2_500_000
    selected = group[group['time'] >= time_cutoff]
    if not selected.empty:
        stats = {
            "variable": variable,
            "key": key,
            "multiplier": multiplier,
            "mean": selected["value"].mean(),
            "median": selected["value"].median(),
            "min": selected["value"].min(),
            "max": selected["value"].max(),
            "std": selected["value"].std(),
        }
        summary_list.append(stats)
summary_table = pd.DataFrame(summary_list)

# Show the entire DataFrame (not truncated)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display_html(summary_table.to_html(index=False), raw=True)